In [1]:
import pandas as pd
import os
import re
from datetime import datetime

def expand_ipc_monthly(csv_path, country_label=None, threshold=0.20,
                       month_start="2017-01", month_end="2025-12"):
    """
    Expand IPC Level-1 dataset into monthly records (2017-01 to 2025-12).
    Keep all admin1 regions and compute Phase 3+ indicators.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"File not found: {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"Raw data: {df.shape[0]} rows / columns: {list(df.columns)}")

    # 1) Remove metadata rows
    mask_meta = df.apply(lambda r: any(str(v).strip().startswith("#") for v in r), axis=1)
    df = df.loc[~mask_meta].copy()

    # 2) Match column names (tolerant mapping)
    cols_lower = {c.lower(): c for c in df.columns}

    def pick(*cands, regex=None):
        for k in cands:
            if k.lower() in cols_lower:
                return cols_lower[k.lower()]
        if regex:
            for c in df.columns:
                if re.search(regex, c, flags=re.IGNORECASE):
                    return c
        return None

    col_analysis = pick("Date of analysis", regex=r"date.*analysis")
    col_country  = pick("Country", "country_iso3", regex=r"country")
    col_lvl1     = pick("Level 1", regex=r"level\s*1|admin.?1|region|state")
    col_valid    = pick("Validity period", regex=r"validity")
    col_from     = pick("From", regex=r"^from$")
    col_to       = pick("To", regex=r"^to$")
    col_phase    = pick("Phase", regex=r"phase")
    col_num      = pick("Number", regex=r"number|affected.*num")
    col_pct      = pick("Percentage", regex=r"percent|pct")

    req = [col_country, col_lvl1, col_from, col_to, col_phase]
    if any(c is None for c in req):
        miss = [n for n,c in zip(["country","level1","from","to","phase"], req) if c is None]
        raise ValueError(f"Missing columns: {miss}")

    # 3) Rename columns
    rename_map = {}
    if col_analysis: rename_map[col_analysis] = "analysis_date_raw"
    rename_map[col_country] = "country"
    rename_map[col_lvl1]    = "admin1"
    if col_valid: rename_map[col_valid] = "validity"
    rename_map[col_from]    = "date_from"
    rename_map[col_to]      = "date_to"
    rename_map[col_phase]   = "phase"
    if col_num: rename_map[col_num] = "affected_num"
    if col_pct: rename_map[col_pct] = "affected_pct"
    df = df.rename(columns=rename_map)

    # 4) Type conversion
    for c in ["affected_num", "affected_pct"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if "affected_pct" in df.columns:
        df.loc[df["affected_pct"] > 1.0, "affected_pct"] /= 100.0

    df["date_from"] = pd.to_datetime(df["date_from"], errors="coerce")
    df["date_to"]   = pd.to_datetime(df["date_to"], errors="coerce")

    if "analysis_date_raw" in df.columns:
        df["analysis_date"] = pd.to_datetime(df["analysis_date_raw"], format="%b %Y", errors="coerce")
    else:
        df["analysis_date"] = pd.NaT

    df["admin1"]  = df["admin1"].astype(str).str.strip()
    df["country"] = country_label or (df["country"].iloc[0] if "country" in df.columns else None)

    # 5) Expand into monthly records
    records = []
    for _, r in df.iterrows():
        if pd.isna(r["date_from"]) or pd.isna(r["date_to"]):
            continue
        for p in pd.period_range(r["date_from"], r["date_to"], freq="M"):
            records.append({
                "country": r["country"],
                "admin1": r["admin1"],
                "year_month": f"{p.year}-{p.month:02d}",
                "analysis_date": r["analysis_date"],
                "validity": r.get("validity"),
                "phase": str(r["phase"]).strip().replace("Phase ", ""),
                "affected_num": r.get("affected_num", pd.NA),
                "affected_pct": r.get("affected_pct", pd.NA),
            })

    long = pd.DataFrame.from_records(records)
    print(f"Expanded monthly records: {long.shape[0]} rows")

    # Keep latest analysis per month
    if "analysis_date" in long.columns:
        long = (long.sort_values(["country","admin1","year_month","analysis_date"])
                    .drop_duplicates(["country","admin1","year_month","phase"], keep="last"))

    # 6) Aggregate Phase 3+
    def is_3plus(ph):
        ph = str(ph).strip()
        return ph in {"3+","3","4","5"}

    mask_3p = long["phase"].map(is_3plus)
    agg = (long[mask_3p]
           .groupby(["country","admin1","year_month"], as_index=False)
           .agg(affected_num_3p=("affected_num","sum"),
                affected_pct_3p=("affected_pct","sum")))

    # 7) Create full month × admin1 grid
    all_admin1 = df["admin1"].dropna().unique()
    all_months = [f"{p.year}-{p.month:02d}" for p in pd.period_range(month_start, month_end, freq="M")]
    country_code = df["country"].iloc[0]

    complete = pd.MultiIndex.from_product(
        [[country_code], all_admin1, all_months],
        names=["country","admin1","year_month"]
    ).to_frame(index=False)

    out = complete.merge(agg, on=["country","admin1","year_month"], how="left")

    # 8) Fill missing values
    out = out.sort_values(["country","admin1","year_month"])
    for col in ["affected_num_3p","affected_pct_3p"]:
        out[col] = out.groupby(["country","admin1"])[col].transform(lambda s: s.ffill().bfill())

    # Estimate admin1 population
    out["admin1_pop_est_raw"] = out.apply(
        lambda r: (r["affected_num_3p"]/r["affected_pct_3p"]) if pd.notna(r["affected_pct_3p"]) and r["affected_pct_3p"]>0 else pd.NA,
        axis=1
    )
    med = (out.groupby(["country","admin1"])["admin1_pop_est_raw"].median()
           .rename("admin1_pop_median").reset_index())
    out = out.merge(med, on=["country","admin1"], how="left")
    out["admin1_pop_est"] = out["admin1_pop_est_raw"].fillna(out["admin1_pop_median"])
    out.drop(columns=["admin1_pop_est_raw","admin1_pop_median"], inplace=True)

    # 9) Binary label for Phase 3+ >= threshold
    out["phase3plus_binary_20pct"] = (out["affected_pct_3p"] >= threshold).astype("Int64")

    # 10) Final columns
    out = out[[
        "country","admin1","year_month",
        "admin1_pop_est","affected_num_3p","affected_pct_3p",
        "phase3plus_binary_20pct"
    ]].sort_values(["country","admin1","year_month"]).reset_index(drop=True)

    return out


def main():
    input_dir = "data raw"
    output_dir = "data clean"
    os.makedirs(output_dir, exist_ok=True)

    countries = {
        "SOM": "ipc_som_level1_long.csv",
        "SSD": "ipc_ssd_level1_long.csv",
        "CAF": "ipc_caf_level1_long.csv",
        "MDG": "ipc_mdg_level1_long.csv",
        "MOZ": "ipc_moz_level1_long.csv",
    }

    results = {}

    print("—" * 40)
    print("Processing IPC Level-1 (2017–2025)")
    print("—" * 40)

    for code, filename in countries.items():
        print(f"\n{code}: {filename}")
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, f"ipc_{code.lower()}_clean.csv")

        try:
            clean_df = expand_ipc_monthly(input_path, country_label=code)
            clean_df.to_csv(output_path, index=False, encoding="utf-8-sig")
            results[code] = clean_df
            print(f"  ✓ Saved to {output_path}")
        except FileNotFoundError as e:
            print(f"  ✗ File not found: {e}")
        except Exception as e:
            print(f"  ✗ Error: {e}")

    print("\n" + "—" * 40)
    print("Summary")
    print("—" * 40)

    for code, df in results.items():
        n_admin1 = df["admin1"].nunique()
        date_range = f"{df['year_month'].min()} to {df['year_month'].max()}"
        months_total = df["year_month"].nunique()
        months_per_admin = len(df) / max(n_admin1, 1)
        phase3_rate = df["phase3plus_binary_20pct"].astype(float).mean() * 100

        print(f"\n{code}: {len(df):,} rows, {n_admin1} admin1s")
        print(f"  Range: {date_range} ({months_total} months)")
        print(f"  ~{months_per_admin:.0f} months per admin1, Phase3+≥20%: {phase3_rate:.1f}%")

    print("\nAll cleaned files saved to:", output_dir)


if __name__ == "__main__":
    main()


————————————————————————————————————————
Processing IPC Level-1 (2017–2025)
————————————————————————————————————————

SOM: ipc_som_level1_long.csv
Raw data: 5835 rows / columns: ['Date of analysis', 'Country', 'Total country population', 'Level 1', 'Validity period', 'From', 'To', 'Phase', 'Number', 'Percentage']
Expanded monthly records: 16380 rows


C:\Users\migo8\AppData\Local\Temp\ipykernel_29072\927198372.py:143: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["admin1_pop_est"] = out["admin1_pop_est_raw"].fillna(out["admin1_pop_median"])


  ✓ Saved to data clean\ipc_som_clean.csv

SSD: ipc_ssd_level1_long.csv
Raw data: 2780 rows / columns: ['Date of analysis', 'Country', 'Total country population', 'Level 1', 'Validity period', 'From', 'To', 'Phase', 'Number', 'Percentage']
Expanded monthly records: 7812 rows
  ✓ Saved to data clean\ipc_ssd_clean.csv

CAF: ipc_caf_level1_long.csv
Raw data: 2479 rows / columns: ['Date of analysis', 'Country', 'Total country population', 'Level 1', 'Validity period', 'From', 'To', 'Phase', 'Number', 'Percentage']
Expanded monthly records: 13853 rows
  ✓ Saved to data clean\ipc_caf_clean.csv

MDG: ipc_mdg_level1_long.csv
Raw data: 1575 rows / columns: ['Date of analysis', 'Country', 'Total country population', 'Level 1', 'Validity period', 'From', 'To', 'Phase', 'Number', 'Percentage']
Expanded monthly records: 6024 rows
  ✓ Saved to data clean\ipc_mdg_clean.csv

MOZ: ipc_moz_level1_long.csv
Raw data: 1429 rows / columns: ['Date of analysis', 'Country', 'Total country population', 'Level 1

C:\Users\migo8\AppData\Local\Temp\ipykernel_29072\927198372.py:143: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["admin1_pop_est"] = out["admin1_pop_est_raw"].fillna(out["admin1_pop_median"])


Expanded monthly records: 7427 rows
  ✓ Saved to data clean\ipc_moz_clean.csv

————————————————————————————————————————
Summary
————————————————————————————————————————

SOM: 2,916 rows, 27 admin1s
  Range: 2017-01 to 2025-12 (108 months)
  ~108 months per admin1, Phase3+≥20%: 87.3%

SSD: 1,512 rows, 14 admin1s
  Range: 2017-01 to 2025-12 (108 months)
  ~108 months per admin1, Phase3+≥20%: 99.3%

CAF: 3,132 rows, 29 admin1s
  Range: 2017-01 to 2025-12 (108 months)
  ~108 months per admin1, Phase3+≥20%: 100.0%

MDG: 2,268 rows, 21 admin1s
  Range: 2017-01 to 2025-12 (108 months)
  ~108 months per admin1, Phase3+≥20%: 87.0%

MOZ: 2,160 rows, 20 admin1s
  Range: 2017-01 to 2025-12 (108 months)
  ~108 months per admin1, Phase3+≥20%: 70.8%

All cleaned files saved to: data clean


C:\Users\migo8\AppData\Local\Temp\ipykernel_29072\927198372.py:143: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["admin1_pop_est"] = out["admin1_pop_est_raw"].fillna(out["admin1_pop_median"])


In [2]:
import pandas as pd
import os

output_dir = "data clean"

# all countries
paths = [
    os.path.join(output_dir, "ipc_som_clean.csv"),
    os.path.join(output_dir, "ipc_ssd_clean.csv"),
    os.path.join(output_dir, "ipc_caf_clean.csv"),
    os.path.join(output_dir, "ipc_mdg_clean.csv"),
    os.path.join(output_dir, "ipc_moz_clean.csv"),
]

dfs = []
for p in paths:
    if os.path.exists(p):
        print(f"Reading {p}")
        dfs.append(pd.read_csv(p))
    else:
        print(f"File not found: {p}")

if not dfs:
    raise ValueError("No IPC files found. Please run the previous step first.")

ipc_all = pd.concat(dfs, ignore_index=True)

print("\n" + "—" * 40)
print("Merged IPC dataset summary")
print("—" * 40)
print(f"Total rows: {ipc_all.shape[0]}")
print(ipc_all.groupby("country")["admin1"].nunique())

merged_path = os.path.join(output_dir, "ipc_all_clean.csv")
ipc_all.to_csv(merged_path, index=False, encoding="utf-8-sig")
print(f"\n✓ Saved merged IPC file: {merged_path}")


Reading data clean\ipc_som_clean.csv
Reading data clean\ipc_ssd_clean.csv
Reading data clean\ipc_caf_clean.csv
Reading data clean\ipc_mdg_clean.csv
Reading data clean\ipc_moz_clean.csv

————————————————————————————————————————
Merged IPC dataset summary
————————————————————————————————————————
Total rows: 11988
country
CAF    29
MDG    21
MOZ    20
SOM    27
SSD    14
Name: admin1, dtype: int64

✓ Saved merged IPC file: data clean\ipc_all_clean.csv


In [3]:
import pandas as pd
import re
from pathlib import Path

# === Paths ===
SOM_DIR = Path("data raw/chirps/som")
SSD_DIR = Path("data raw/chirps/ssd")
CAF_DIR = Path("data raw/chirps/caf")
MDG_DIR = Path("data raw/chirps/mdg")
MOZ_DIR = Path("data raw/chirps/moz")

OUTPUT_DIR = Path("data clean")
OUTPUT_DIR.mkdir(exist_ok=True)

PREFER_PRELIM = True  # True = Prelim overwrites same-year final

# === Country name mapping ===
COUNTRY_TO_ISO3 = {
    "Somalia": "SOM",
    "South Sudan": "SSD",
    "Central African Republic": "CAF",
    "CAR": "CAF",
    "S. Sudan": "SSD",
    "Som": "SOM",
    "Madagascar": "MDG",
    "Mozambique": "MOZ",
}

# === Helper functions ===
def make_join_key(txt: str) -> str:
    """Normalize text for join key (lowercase, remove symbols/spaces)."""
    if pd.isna(txt):
        return txt
    x = str(txt).lower().strip()
    x = re.sub(r"[-_]+", " ", x)
    x = " ".join(x.split())
    x = re.sub(r"[^a-z0-9 ]+", "", x)
    return x.replace(" ", "")

def parse_country_admin_from_stem(stem: str):
    """Extract (country, admin1, join key) from filename."""
    m = re.match(r"^(?P<country>.+?)\+(?P<admin1>.+?)(?:_Monthly)?$", stem)
    if not m:
        return None, None, None
    country_raw = m.group("country").strip()
    admin1_display = m.group("admin1").replace("_", " ").strip()
    country_iso3 = COUNTRY_TO_ISO3.get(country_raw, country_raw)
    admin1_key = make_join_key(admin1_display)
    return country_iso3, admin1_display, admin1_key

def collapse_prelim_into_year(df_wide: pd.DataFrame, prefer_prelim: bool = True) -> pd.DataFrame:
    """Merge 'YYYY Prelim' columns into final year."""
    df = df_wide.copy()
    year_cols = [c for c in df.columns if re.fullmatch(r"\d{4}", str(c))]
    prelim_cols = [c for c in df.columns if re.fullmatch(r"\d{4}\s+Prelim", str(c))]
    for prelim in prelim_cols:
        year = prelim.replace(" Prelim", "")
        if year not in df.columns:
            df.rename(columns={prelim: year}, inplace=True)
        else:
            if prefer_prelim:
                df[year] = df[prelim].combine_first(df[year])
            else:
                df[year] = df[year].combine_first(df[prelim])
            df.drop(columns=[prelim], inplace=True)
    return df

def month_to_num(m: str) -> int:
    """Convert month name (full or abbrev) to number."""
    m = str(m).strip().title()[:3]
    mapping = {
        "Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
        "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12
    }
    return mapping.get(m, pd.NA)

def infer_calendar_year(year_raw: str, month_num: int):
    """
    Convert 'YYYY' or 'YYYY-YYYY' to calendar year.
    Southern Hemisphere rule (like MDG/MOZ):
      - '2024-2025':
          Oct–Dec → 2024 (start year)
          Jan–Sep → 2025 (end year)
    """
    s = str(year_raw).strip()
    if re.fullmatch(r"\d{4}", s):
        return int(s)
    m = re.findall(r"\d{4}", s)
    if len(m) == 2:
        start, end = map(int, m)
        return start if month_num in (10,11,12) else end
    return pd.NA

def load_rainfall_from_folder(folder: Path, prefer_prelim: bool = True) -> pd.DataFrame:
    """Load CHIRPS CSVs for a country and return long-form monthly data."""
    files = list(folder.glob("*.csv")) + list(folder.glob("*.CSV"))
    if not files:
        files = list(folder.rglob("*.csv")) + list(folder.rglob("*.CSV"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {folder}")

    all_rows = []
    for f in files:
        country_iso3, admin1_display, admin1_key = parse_country_admin_from_stem(f.stem)
        if country_iso3 is None:
            print(f"  [SKIP] Unrecognized filename: {f.name}")
            continue

        wide = pd.read_csv(f, encoding="utf-8", encoding_errors="ignore")
        wide = wide.loc[:, ~wide.columns.str.contains(r"^Unnamed")].copy()
        wide = collapse_prelim_into_year(wide, prefer_prelim=prefer_prelim)

        year_cols = [c for c in wide.columns if re.fullmatch(r"\d{4}(\-\d{4})?", str(c))]
        if "Month" not in wide.columns:
            raise ValueError(f"{f.name} missing 'Month' column")
        if "Period" not in wide.columns:
            wide["Period"] = pd.NA

        long = wide.melt(
            id_vars=["Month", "Period"],
            value_vars=year_cols,
            var_name="YearRaw",
            value_name="rain_mm",
        )

        long["mm"] = long["Month"].map(month_to_num)
        long = long.dropna(subset=["mm"]).copy()
        long["mm"] = long["mm"].astype(int)
        long["year"] = long.apply(lambda r: infer_calendar_year(r["YearRaw"], r["mm"]), axis=1)
        long = long.dropna(subset=["year"]).copy()
        long["year"] = long["year"].astype(int)
        long["year_month"] = long["year"].astype(str) + "-" + long["mm"].astype(str).str.zfill(2)
        long["country"] = country_iso3
        long["admin1"] = admin1_display
        long["admin1_key"] = admin1_key

        all_rows.append(long[["country", "admin1", "admin1_key", "year_month", "rain_mm"]])

    rain = pd.concat(all_rows, ignore_index=True)
    rain = rain.sort_values(["country", "admin1", "year_month"]).reset_index(drop=True)

    dup_mask = rain.duplicated(subset=["country", "admin1_key", "year_month"], keep=False)
    if dup_mask.any():
        print(f"  [WARN] {dup_mask.sum()} duplicated (country, admin1_key, year_month). Averaging.")
        rain = (
            rain.groupby(["country", "admin1", "admin1_key", "year_month"], as_index=False)
                .agg(rain_mm=("rain_mm", "mean"))
        )

    rain = rain.drop_duplicates(subset=["country", "admin1_key", "year_month"], keep="first")
    return rain

def expand_to_full_timeline_with_climatology(df: pd.DataFrame,
                                             start_year: int = 2017,
                                             end_year: int = 2025) -> pd.DataFrame:
    """Fill missing months using same-month climatology mean."""
    all_months = [f"{p.year}-{p.month:02d}" for p in pd.period_range(f"{start_year}-01", f"{end_year}-12", freq="M")]
    combos = df[["country", "admin1", "admin1_key"]].drop_duplicates()

    frame = pd.DataFrame(
        [{"country": r["country"], "admin1": r["admin1"], "admin1_key": r["admin1_key"], "year_month": ym}
         for _, r in combos.iterrows() for ym in all_months]
    )

    out = frame.merge(df, on=["country", "admin1", "admin1_key", "year_month"], how="left")
    out["mm"] = out["year_month"].str[-2:].astype(int)

    clim = (
        out.groupby(["country", "admin1_key", "mm"], as_index=False)
        .agg(clim_mm=("rain_mm", "mean"))
    )

    out = out.merge(clim, on=["country", "admin1_key", "mm"], how="left")
    out["rain_mm"] = out["rain_mm"].fillna(out["clim_mm"])
    out = out.drop(columns=["mm", "clim_mm"]).sort_values(["country", "admin1", "year_month"]).reset_index(drop=True)
    return out

# === Main ===
def main():
    print("—" * 40)
    print("Processing CHIRPS rainfall data")
    print("—" * 40)

    countries = {
        "SOM": SOM_DIR, "SSD": SSD_DIR, "CAF": CAF_DIR,
        "MDG": MDG_DIR, "MOZ": MOZ_DIR  # added Mozambique
    }

    all_data, results = [], {}
    for code, folder in countries.items():
        print(f"\n{code}: {folder}")
        try:
            rain = load_rainfall_from_folder(folder, prefer_prelim=PREFER_PRELIM)
            print(f"  Raw: {rain.shape[0]} rows, {rain['admin1'].nunique()} admin1s")

            rain_full = expand_to_full_timeline_with_climatology(rain)
            print(f"  Full timeline: {rain_full.shape[0]} rows")

            output_path = OUTPUT_DIR / f"chirps_{code.lower()}_admin1_monthly.csv"
            rain_full.to_csv(output_path, index=False, encoding="utf-8-sig")
            print(f"   Saved: {output_path}")

            all_data.append(rain_full)
            results[code] = rain_full
        except FileNotFoundError as e:
            print(f"   File not found: {e}")
        except Exception as e:
            print(f"   Error: {e}")

    if all_data:
        rain_all = pd.concat(all_data, ignore_index=True)
        out_all = OUTPUT_DIR / "chirps_all_admin1_monthly.csv"
        rain_all.to_csv(out_all, index=False, encoding="utf-8-sig")
        print(f"\n Saved merged file: {out_all}")

    print("\n" + "—" * 40)
    print("Summary")
    print("—" * 40)
    for code, df in results.items():
        n_admin1 = df["admin1"].nunique()
        date_range = f"{df['year_month'].min()} to {df['year_month'].max()}"
        avg_rain = df["rain_mm"].mean()
        months_per_admin = len(df) / max(n_admin1, 1)
        print(f"\n{code}: {len(df):,} rows, {n_admin1} admin1s")
        print(f"  Range: {date_range}, ~{months_per_admin:.0f} months/admin1")
        print(f"  Avg rainfall: {avg_rain:.1f} mm")

    print("\nAll outputs saved to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()


————————————————————————————————————————
Processing CHIRPS rainfall data
————————————————————————————————————————

SOM: data raw\chirps\som
  [WARN] 3672 duplicated (country, admin1_key, year_month). Averaging.
  Raw: 1836 rows, 17 admin1s
  Full timeline: 1836 rows
   Saved: data clean\chirps_som_admin1_monthly.csv

SSD: data raw\chirps\ssd
  [WARN] 2160 duplicated (country, admin1_key, year_month). Averaging.
  Raw: 1080 rows, 10 admin1s
  Full timeline: 1080 rows
   Saved: data clean\chirps_ssd_admin1_monthly.csv

CAF: data raw\chirps\caf
  [WARN] 3456 duplicated (country, admin1_key, year_month). Averaging.
  Raw: 1728 rows, 16 admin1s
  Full timeline: 1728 rows
   Saved: data clean\chirps_caf_admin1_monthly.csv

MDG: data raw\chirps\mdg
  [WARN] 1296 duplicated (country, admin1_key, year_month). Averaging.
  Raw: 648 rows, 6 admin1s
  Full timeline: 648 rows
   Saved: data clean\chirps_mdg_admin1_monthly.csv

MOZ: data raw\chirps\moz
  [WARN] 2160 duplicated (country, admin1_key, 

In [4]:
import pandas as pd
from pathlib import Path

# === Paths ===
IPC_PATH = Path("data clean/ipc_all_clean.csv")
RAIN_PATH = Path("data clean/chirps_all_admin1_monthly.csv")
OUT_PATH  = Path("data clean/admin1_mapping_check.csv")

# === Load data ===
ipc = pd.read_csv(IPC_PATH, usecols=["country", "admin1"])
rain = pd.read_csv(RAIN_PATH, usecols=["country", "admin1"])

# === Normalize join keys ===
def make_key(x):
    import re
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9]+", "", x)
    return x

ipc["admin1_key"] = ipc["admin1"].map(make_key)
rain["admin1_key"] = rain["admin1"].map(make_key)

# === Compare ===
merged = ipc.merge(
    rain[["country", "admin1_key", "admin1"]],
    on=["country", "admin1_key"], how="outer",
    suffixes=("_ipc", "_rain")
)

# === Find unmatched ===
unmatched = merged[merged["admin1_ipc"].isna() | merged["admin1_rain"].isna()]
unmatched = unmatched.sort_values(["country", "admin1_key"])

print(f"Found {len(unmatched)} unmatched admin1s")
unmatched.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved mismatch list: {OUT_PATH}")


Found 8100 unmatched admin1s
Saved mismatch list: data clean\admin1_mapping_check.csv


In [5]:
# ======================================================
# harmonize_rainfall.py — IPC × CHIRPS
# ======================================================

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

# === Paths ===
DATA_CLEAN = Path("data clean")
IPC_PATH   = DATA_CLEAN / "ipc_all_clean.csv"
RAIN_PATH  = DATA_CLEAN / "chirps_all_admin1_monthly.csv"
OUT_PATH   = DATA_CLEAN / "chirps_all_admin1_monthly_harmonized.csv"

# === Helper ===
def make_join_key(txt: str) -> str:
    """Normalize string for joining."""
    if pd.isna(txt):
        return txt
    x = str(txt).lower().strip()
    x = re.sub(r"[’'`]", "", x)
    x = re.sub(r"[-_]+", " ", x)
    x = " ".join(x.split())
    x = re.sub(r"[^a-z0-9 ]+", "", x)
    return x.replace(" ", "")

def normalize_alias(d: dict) -> dict:
    """Normalize dict keys and values."""
    return {make_join_key(k): make_join_key(v) for k, v in d.items()}

def normalize_alias_to_value(d: dict) -> dict:
    """Normalize keys only, keep original values."""
    return {make_join_key(k): v for k, v in d.items()}

# === Country code map ===
COUNTRY_TO_ISO3 = {
    "Somalia": "SOM",
    "South Sudan": "SSD",
    "Central African Republic": "CAF",
    "CAR": "CAF",
    "S. Sudan": "SSD",
    "Som": "SOM",
    "Madagascar": "MDG",
    "Mozambique": "MOZ",
}

# === Somalia ===
SOM_ALIAS = normalize_alias({
    "Juba Dhexe": "Middle Juba", "Juba Hoose": "Lower Juba",
    "Shabelle Dhexe": "Middle Shabelle", "Shabelle Hoose": "Lower Shabelle",
    "Woqooyi Galbeed": "Woqooyi Galbeed",
})

# === South Sudan ===
SSD_COMPOSITES = {
    make_join_key("Jonglei & Pibor Administrative Area"): [make_join_key("Jonglei"), make_join_key("Pibor")],
    make_join_key("Unity & Ruweng Administrative Area"):  [make_join_key("Unity"),  make_join_key("Ruweng")],
}
SSD_ALIAS = {}

# === Central African Republic ===
CAF_ALIAS = normalize_alias({
    "Bangui Ville": "Bangui", "Lim-Pendé": "Ouham-Pendé", "Ouham-Fafa": "Ouham",
    "Nana-Mambéré": "Nana-Mambéré", "Mambéré-Kadéï": "Mambéré-Kadéï",
    "Haut-Oubangui": "Haut-Mbomou", "Equateur": "Lobaye", "Kagas": "Vakaga",
})
CAF_EXCLUDE = {make_join_key(x) for x in [
    "Prefectures", "Concentrations in Prefectures analyzed", "Fertit", "Plateaux", "Yadé"
]}

# === Madagascar (region → province) ===
MDG_REGION_TO_PROVINCE = normalize_alias_to_value({
    "Diana": "Antsiranana", "Sava": "Antsiranana",
    "Analanjirofo": "Toamasina", "Atsinanana": "Toamasina", "Alaotra Mangoro": "Toamasina",
    "Analamanga": "Antananarivo", "Itasy": "Antananarivo", "Vakinankaratra": "Antananarivo", "Bongolava": "Antananarivo",
    "Vatovavy": "Fianarantsoa", "Fitovinany": "Fianarantsoa", "Fitovivany": "Fianarantsoa",
    "VATOVAVY FITOVINANY": "Fianarantsoa", "Vatovavy Fitovinany": "Fianarantsoa",
    "Haute Matsiatra": "Fianarantsoa", "Amoron'i Mania": "Fianarantsoa", "Ihorombe": "Fianarantsoa",
    "Atsimo Atsinanana": "Fianarantsoa", "Atsimo-Atsinana": "Fianarantsoa",
    "Boeny": "Mahajanga", "Betsiboka": "Mahajanga", "Melaky": "Mahajanga", "Sofia": "Mahajanga",
    "Androy": "Toliara", "Anosy": "Toliara", "Atsimo Andrefana": "Toliara", "Menabe": "Toliara",
})
_MDG_PROVINCE_KEYS = {make_join_key(p) for p in
    {"Antananarivo","Antsiranana","Fianarantsoa","Mahajanga","Toamasina","Toliara"}}

def mdg_region_to_province_key(name: str) -> str:
    if pd.isna(name): return ""
    k = make_join_key(name)
    province = MDG_REGION_TO_PROVINCE.get(k)
    if province: return make_join_key(province)
    if k in _MDG_PROVINCE_KEYS: return k
    return ""

# === Mozambique (region → province) ===
MOZ_REGION_TO_PROVINCE = normalize_alias_to_value({
    "Acolhedores": "Maputo",
    "Centro": "Sofala",
    "Deslocados": "Sofala",
    "Maputo Cidade": "Maputo",
    "Maputo City": "Maputo",
    "Maputo Provincia": "Maputo",
    "Maputo e Matola": "Maputo",
    "Provincia de Maputo": "Maputo",
    "Sul": "Gaza",
    "Zambézia": "Zambezia",
})
_MOZ_PROVINCE_KEYS = {make_join_key(p) for p in
    {"Cabo Delgado","Gaza","Inhambane","Manica","Maputo",
     "Nampula","Niassa","Sofala","Tete","Zambezia"}}

def moz_region_to_province_key(name: str) -> str:
    if pd.isna(name): return ""
    k = make_join_key(name)
    province = MOZ_REGION_TO_PROVINCE.get(k)
    if province: return make_join_key(province)
    if k in _MOZ_PROVINCE_KEYS: return k
    return ""

# === Apply alias ===
def apply_alias(country, admin1_key):
    if country == "SOM":
        return SOM_ALIAS.get(admin1_key, admin1_key)
    if country == "SSD":
        return SSD_ALIAS.get(admin1_key, admin1_key)
    if country == "CAF":
        if admin1_key in CAF_EXCLUDE:
            return np.nan
        return CAF_ALIAS.get(admin1_key, admin1_key)
    return admin1_key

# === Main ===
def main():
    print("—" * 40)
    print("Aligning CHIRPS rainfall data with IPC regions")
    print("—" * 40)

    for p in [IPC_PATH, RAIN_PATH]:
        if not p.exists():
            raise FileNotFoundError(f"Missing file: {p}")

    ipc  = pd.read_csv(IPC_PATH, dtype=str)
    rain = pd.read_csv(RAIN_PATH, dtype=str)

    ipc["country"]  = ipc["country"].map(COUNTRY_TO_ISO3).fillna(ipc["country"])
    rain["country"] = rain["country"].map(COUNTRY_TO_ISO3).fillna(rain["country"])
    ipc["admin1_key"] = ipc["admin1"].apply(make_join_key)
    rain["admin1_key"] = rain["admin1"].apply(make_join_key)
    rain["rain_mm"] = pd.to_numeric(rain["rain_mm"], errors="coerce")

    dup_mask = rain.duplicated(subset=["country", "admin1_key", "year_month"], keep=False)
    if dup_mask.any():
        print(f"[WARN] {dup_mask.sum()} duplicates in rainfall, averaging...")
        rain = rain.groupby(["country","admin1","admin1_key","year_month"], as_index=False).agg(rain_mm=("rain_mm","mean"))

    # Apply alias
    rain_alias = rain.copy()
    rain_alias["admin1_key_alias"] = rain_alias.apply(
        lambda r: apply_alias(str(r["country"]), str(r["admin1_key"])), axis=1
    )
    rain_alias = rain_alias[~rain_alias["admin1_key_alias"].isna()].copy()
    rain_simple = (
        rain_alias.groupby(["country","admin1_key_alias","year_month"], as_index=False)
        .agg(rain_mm=("rain_mm","mean"))
        .rename(columns={"admin1_key_alias":"admin1_key"})
    )

    # Build SSD composites
    def build_ssd_composites(rain_df, composites_dict):
        rows = []
        for comp_key, parts in composites_dict.items():
            sub = rain_df[(rain_df["country"]=="SSD") & (rain_df["admin1_key"].isin(parts))] \
                .groupby(["year_month"], as_index=False).agg(rain_mm=("rain_mm","mean"))
            if not sub.empty:
                sub.insert(0,"country","SSD")
                sub.insert(1,"admin1_key",comp_key)
                rows.append(sub)
        return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    ssd_comp = build_ssd_composites(rain_simple, SSD_COMPOSITES)
    rain_harmonized = pd.concat([rain_simple, ssd_comp], ignore_index=True)
    rain_harmonized["join_key"] = rain_harmonized["admin1_key"]

    # IPC base
    ipc_base = ipc[["country","admin1","admin1_key","year_month"]].drop_duplicates().copy()
    ipc_base["join_key"] = ipc_base["admin1_key"]

    # Apply crosswalks
    mdg_mask = ipc_base["country"] == "MDG"
    ipc_base.loc[mdg_mask,"join_key"] = ipc_base.loc[mdg_mask,"admin1"].map(mdg_region_to_province_key)
    moz_mask = ipc_base["country"] == "MOZ"
    ipc_base.loc[moz_mask,"join_key"] = ipc_base.loc[moz_mask,"admin1"].map(moz_region_to_province_key)

    # Unmapped checks
    for c, path, label in [
        ("MDG", "mdg_unmapped_regions_for_join_rain.csv", "MDG"),
        ("MOZ", "moz_unmapped_admin1_for_join_rain.csv", "MOZ")
    ]:
        unmapped = ipc_base[(ipc_base["country"]==c) & ipc_base["join_key"].isin(["",None,pd.NA])]["admin1"].drop_duplicates()
        if len(unmapped):
            out_path = DATA_CLEAN / path
            unmapped.to_csv(out_path, index=False, encoding="utf-8-sig")
            print(f"[{label}] {len(unmapped)} regions not mapped → saved to {out_path}")
        else:
            print(f"[{label}] All IPC admin1s matched rainfall")

    # Merge
    ml = ipc_base.merge(
        rain_harmonized[["country","join_key","year_month","rain_mm"]],
        on=["country","join_key","year_month"], how="left"
    )

    # Fill missing rainfall by same-country mean and climatology
    ml["rain_mm"] = ml.groupby(["country","year_month"])["rain_mm"].transform(lambda s: s.fillna(s.mean()))
    ml["mm"] = ml["year_month"].str[-2:].astype(int)
    clim = ml.groupby(["country","mm"],as_index=False).agg(rain_clim=("rain_mm","mean"))
    ml = ml.merge(clim,on=["country","mm"],how="left")
    ml["rain_mm"] = ml["rain_mm"].fillna(ml["rain_clim"])
    ml.drop(columns=["mm","rain_clim"],inplace=True)

    # Remove duplicate keys
    dup_mask2 = ml.duplicated(subset=["country","admin1_key","year_month"],keep=False)
    if dup_mask2.any():
        print(f"[WARN] {dup_mask2.sum()} duplicates before export, averaging again...")
        ml = ml.groupby(["country","admin1","admin1_key","year_month"],as_index=False).agg(rain_mm=("rain_mm","mean"))

    ml.to_csv(OUT_PATH,index=False,encoding="utf-8-sig")
    print(f"\nSaved: {OUT_PATH} ({len(ml)} rows)")

    # Coverage summary
    ipc_cnt = ipc[["country","admin1_key"]].drop_duplicates().groupby("country").size().rename("ipc_admin1_cnt")
    rain_cnt_before = rain.groupby("country")["admin1_key"].nunique().rename("rain_admin1_cnt_raw")
    ml_nonnull = ml[~ml["rain_mm"].isna()]
    rain_cnt_after = ml_nonnull.groupby("country")["admin1_key"].nunique().rename("admin1_cnt_after")
    rep = pd.concat([ipc_cnt, rain_cnt_before, rain_cnt_after],axis=1).reset_index()
    rep["coverage_ratio"] = (rep["admin1_cnt_after"]/rep["ipc_admin1_cnt"]).round(3)

    print("—" * 40)
    print("Coverage summary (admin1 counts)")
    print("—" * 40)
    print(rep.to_string(index=False))

if __name__ == "__main__":
    main()


————————————————————————————————————————
Aligning CHIRPS rainfall data with IPC regions
————————————————————————————————————————
[MDG] All IPC admin1s matched rainfall
[MOZ] All IPC admin1s matched rainfall
[WARN] 2700 duplicates before export, averaging again...

Saved: data clean\chirps_all_admin1_monthly_harmonized.csv (11988 rows)
————————————————————————————————————————
Coverage summary (admin1 counts)
————————————————————————————————————————
country  ipc_admin1_cnt  rain_admin1_cnt_raw  admin1_cnt_after  coverage_ratio
    CAF              29                   16                29             1.0
    MDG              13                    6                13             1.0
    MOZ              20                   10                20             1.0
    SOM              22                   17                22             1.0
    SSD              14                   10                14             1.0


In [6]:
# ======================================================
# build_conflict_features.py — Harmonize ACLED conflict data to IPC admin1
# ======================================================

import re
import pandas as pd
from pathlib import Path

# === Paths ===
DATA_RAW = Path("data raw")
DATA_CLEAN = Path("data clean")
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

ACLED_XLSX = DATA_RAW / "ACLED_Africa_aggregated_data.xlsx"
IPC_PATH = DATA_CLEAN / "ipc_all_clean.csv"
OUT_PATH = DATA_CLEAN / "conflict_events_admin1_monthly.csv"

START_YM, END_YM = "2017-01", "2025-12"

# === Country mapping ===
COUNTRY_TO_ISO3 = {
    "Somalia": "SOM",
    "South Sudan": "SSD",
    "Central African Republic": "CAF",
    "CAR": "CAF",
    "Madagascar": "MDG",
    "Mozambique": "MOZ",
}
TARGET_ISO3 = {"SOM", "SSD", "CAF", "MDG", "MOZ"}

# === Utility functions ===
def make_join_key(txt: str) -> str:
    if pd.isna(txt):
        return txt
    x = str(txt).lower().strip()
    x = re.sub(r"[’'`]", "", x)
    x = re.sub(r"[-_]+", " ", x)
    x = " ".join(x.split())
    x = re.sub(r"[^a-z0-9 ]+", "", x)
    return x.replace(" ", "")

def build_full_months(start_ym: str, end_ym: str):
    pr = pd.period_range(start_ym, end_ym, freq="M")
    return [f"{p.year}-{p.month:02d}" for p in pr]

# === Somalia alias normalization ===
SOM_ALIAS = {
    "Banadir": "Banadir",
    "Juba Dhexe": "Middle Juba",
    "Juba Hoose": "Lower Juba",
    "Shabelle Dhexe": "Middle Shabelle",
    "Shabelle Hoose": "Lower Shabelle",
}
_SOM_ALIAS_NORM = {make_join_key(k): v for k, v in SOM_ALIAS.items()}

def som_normalize_admin1(name: str) -> str:
    raw = str(name).strip()
    k = make_join_key(raw)
    return _SOM_ALIAS_NORM.get(k, raw)

# === Madagascar region → province crosswalk ===
def normalize_alias(d: dict) -> dict:
    return {make_join_key(k): v for k, v in d.items()}

MDG_REGION_TO_PROVINCE = normalize_alias({
    "Diana": "Antsiranana", "Sava": "Antsiranana",
    "Analanjirofo": "Toamasina", "Atsinanana": "Toamasina", "Alaotra Mangoro": "Toamasina",
    "Analamanga": "Antananarivo", "Itasy": "Antananarivo", "Vakinankaratra": "Antananarivo", "Bongolava": "Antananarivo",
    "Vatovavy": "Fianarantsoa", "Fitovinany": "Fianarantsoa", "Fitovivany": "Fianarantsoa",
    "Haute Matsiatra": "Fianarantsoa", "Amoron'i Mania": "Fianarantsoa", "Ihorombe": "Fianarantsoa",
    "Atsimo Atsinanana": "Fianarantsoa", "Atsimo-Atsinana": "Fianarantsoa",
    "VATOVAVY FITOVINANY": "Fianarantsoa", "Vatovavy Fitovinany": "Fianarantsoa",
    "Boeny": "Mahajanga", "Betsiboka": "Mahajanga", "Melaky": "Mahajanga", "Sofia": "Mahajanga",
    "Androy": "Toliara", "Anosy": "Toliara", "Atsimo Andrefana": "Toliara", "Menabe": "Toliara",
})
_MDG_PROVINCE_KEYS = {make_join_key(p) for p in
    {"Antananarivo", "Antsiranana", "Fianarantsoa", "Mahajanga", "Toamasina", "Toliara"}}

def mdg_region_to_province_key(name: str) -> str:
    if pd.isna(name):
        return ""
    k = make_join_key(name)
    k = re.sub(r"\s+", "", k)
    k = k.replace("fitovivany", "fitovinany")
    province = MDG_REGION_TO_PROVINCE.get(k)
    if province:
        return make_join_key(province)
    if k in _MDG_PROVINCE_KEYS:
        return k
    return ""

# === Mozambique region → province crosswalk ===
MOZ_REGION_TO_PROVINCE = normalize_alias({
    "Niassa": "Niassa", "Cabo Delgado": "Cabo Delgado", "Nampula": "Nampula",
    "Zambezia": "Zambezia", "Zambézia": "Zambezia", "Tete": "Tete", "Manica": "Manica",
    "Sofala": "Sofala", "Inhambane": "Inhambane", "Gaza": "Gaza", "Maputo": "Maputo",
    "Maputo Cidade": "Maputo", "Maputo City": "Maputo", "Maputo Provincia": "Maputo",
    "Provincia de Maputo": "Maputo", "Maputo e Matola": "Maputo",
    "Centro": "Sofala", "Sul": "Maputo", "Acolhedores": "Maputo", "Deslocados": "Tete"
})
_MOZ_PROVINCE_KEYS = {make_join_key(p) for p in
    {"Niassa","Cabo Delgado","Nampula","Zambezia","Tete","Manica","Sofala","Inhambane","Gaza","Maputo"}}

def moz_region_to_province_key(name: str) -> str:
    if pd.isna(name):
        return ""
    k = make_join_key(name)
    province = MOZ_REGION_TO_PROVINCE.get(k)
    if province:
        return make_join_key(province)
    if k in _MOZ_PROVINCE_KEYS:
        return k
    return ""

# === Column detection and date parsing ===
def detect_columns(df: pd.DataFrame):
    cl = {c.lower(): c for c in df.columns}
    def pick(*names):
        for n in names:
            if n in cl:
                return cl[n]
        return None
    return {
        "country": pick("country_iso3", "iso3", "country"),
        "admin1": pick("admin1", "adm1", "region", "province", "state"),
        "events": pick("events", "event_count", "count"),
        "fatalities": pick("fatalities", "deaths"),
        "week": pick("week", "week_start", "week_date"),
        "date": pick("event_date", "date", "day"),
        "year": pick("year", "yr"),
        "month": pick("month", "mn", "mo"),
    }

def to_year_month(df: pd.DataFrame, cols: dict) -> pd.Series:
    if cols["week"] is not None:
        s = pd.to_datetime(df[cols["week"]], errors="coerce", dayfirst=True)
        return s.dt.to_period("M").astype(str)
    if cols["date"] is not None:
        s = pd.to_datetime(df[cols["date"]], errors="coerce", dayfirst=True)
        return s.dt.to_period("M").astype(str)
    if cols["year"] and cols["month"]:
        y = pd.to_numeric(df[cols["year"]], errors="coerce").astype("Int64")
        m = pd.to_numeric(df[cols["month"]], errors="coerce").astype("Int64")
        ym = (y.astype("string") + "-" + m.astype("string").str.zfill(2)).astype(object)
        ym[(y.isna()) | (m.isna())] = pd.NA
        return ym
    raise ValueError("Date columns not found (need week, date, or year+month)")

# === Main process ===
def main():
    print("————————————————————————————————————————")
    print("Building conflict features (ACLED → IPC admin1)")
    print("————————————————————————————————————————")

    if not ACLED_XLSX.exists():
        raise FileNotFoundError(f"File not found: {ACLED_XLSX}")

    sheets = pd.read_excel(ACLED_XLSX, sheet_name=None, engine="openpyxl")
    all_rows = []

    for sheet_name, df in sheets.items():
        if df is None or df.empty:
            continue

        cols = detect_columns(df)
        if cols["country"] is None or cols["admin1"] is None:
            continue

        work = df.copy()
        try:
            work["year_month"] = to_year_month(work, cols)
        except Exception:
            continue
        work = work.dropna(subset=["year_month"])

        months = set(build_full_months(START_YM, END_YM))
        work = work[work["year_month"].isin(months)].copy()
        if work.empty:
            continue

        work["country_raw"] = work[cols["country"]].astype(str).str.strip()
        work["country_iso3"] = work["country_raw"].map(COUNTRY_TO_ISO3).fillna(work["country_raw"])
        work = work[work["country_iso3"].isin(TARGET_ISO3)].copy()
        if work.empty:
            continue

        work["admin1"] = work[cols["admin1"]].astype(str).str.strip()
        is_som = work["country_iso3"] == "SOM"
        work.loc[is_som, "admin1"] = work.loc[is_som, "admin1"].apply(som_normalize_admin1)

        work["events"] = pd.to_numeric(work[cols["events"]], errors="coerce").fillna(0)
        work["fatalities"] = pd.to_numeric(work[cols["fatalities"]], errors="coerce").fillna(0)

        work["admin1_key"] = work["admin1"].apply(make_join_key)
        work["join_key"] = work["admin1_key"]

        mdg_mask = work["country_iso3"] == "MDG"
        moz_mask = work["country_iso3"] == "MOZ"
        work.loc[mdg_mask, "join_key"] = work.loc[mdg_mask, "admin1"].map(mdg_region_to_province_key)
        work.loc[moz_mask, "join_key"] = work.loc[moz_mask, "admin1"].map(moz_region_to_province_key)

        agg = (
            work.groupby(["country_iso3", "join_key", "year_month"], as_index=False)
            .agg(conflict_events=("events", "sum"), fatalities=("fatalities", "sum"))
        )
        all_rows.append(agg)

    if not all_rows:
        raise RuntimeError("No valid data found.")

    conflict = pd.concat(all_rows, ignore_index=True).rename(columns={"country_iso3": "country"})

    ipc = pd.read_csv(IPC_PATH, dtype=str)
    ipc["admin1_key"] = ipc["admin1"].apply(make_join_key)
    ipc["join_key"] = ipc["admin1_key"]

    mdg_mask = ipc["country"] == "MDG"
    moz_mask = ipc["country"] == "MOZ"
    ipc.loc[mdg_mask, "join_key"] = ipc.loc[mdg_mask, "admin1"].map(mdg_region_to_province_key)
    ipc.loc[moz_mask, "join_key"] = ipc.loc[moz_mask, "admin1"].map(moz_region_to_province_key)

    all_months = build_full_months(START_YM, END_YM)
    frame = pd.DataFrame(
        [
            {"country": r["country"], "admin1": r["admin1"], "admin1_key": r["admin1_key"],
             "join_key": r["join_key"], "year_month": ym}
            for _, r in ipc.iterrows() for ym in all_months
        ]
    )

    out = frame.merge(conflict, on=["country", "join_key", "year_month"], how="left")
    out["conflict_events"] = out["conflict_events"].fillna(0).astype(int)
    out["fatalities"] = out["fatalities"].fillna(0).astype(int)

    out = (
        out.groupby(["country", "admin1", "admin1_key", "year_month"], as_index=False)
        .agg(conflict_events=("conflict_events", "sum"), fatalities=("fatalities", "sum"))
    )

    out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUT_PATH} ({len(out)} rows)")

    for c in sorted(TARGET_ISO3):
        sub = out[out["country"] == c]
        if len(sub):
            print(f"{c}: admin1={sub['admin1'].nunique()}, months={sub['year_month'].nunique()}, avg_events={sub['conflict_events'].mean():.2f}")

if __name__ == "__main__":
    main()


————————————————————————————————————————
Building conflict features (ACLED → IPC admin1)
————————————————————————————————————————
Saved: data clean\conflict_events_admin1_monthly.csv (11988 rows)
CAF: admin1=29, months=108, avg_events=128.03
MDG: admin1=21, months=108, avg_events=452.57
MOZ: admin1=20, months=108, avg_events=447.85
SOM: admin1=27, months=108, avg_events=1144.89
SSD: admin1=14, months=108, avg_events=894.86


In [7]:
# ======================================================
# harmonize_food_prices.py — WFP price data → unified units & USD
# FX: World Bank API (PA.NUS.FCRF annual) → monthly
# ======================================================

import re
import pandas as pd
import numpy as np
from pathlib import Path
import requests

# === Paths ===
DATA_RAW = Path("data raw")
OUT_DIR = Path("data clean")
OUT_DIR.mkdir(exist_ok=True, parents=True)

OUT_PATH = OUT_DIR / "food_prices_harmonized_common_usd.csv"
FX_CACHE = OUT_DIR / "fx_monthly_usd.csv"

IN_FILES = {
    "SOM": DATA_RAW / "SOM-Prices-Export.csv",
    "SSD": DATA_RAW / "SSD-Prices-Export.csv",
    "CAF": DATA_RAW / "CAF-Prices-Export.csv",
    "MDG": DATA_RAW / "MDG-Prices-Export.csv",
    "MOZ": DATA_RAW / "MOZ-Prices-Export.csv",
}

WB_ISO2 = {"SOM": "SO", "SSD": "SS", "CAF": "CF", "MDG": "MG", "MOZ": "MZ"}
FX_FALLBACK_LCU_PER_USD = {"SOM": 57000, "SSD": 1050, "CAF": 610, "MDG": 4500, "MOZ": 65}

START_YM = "2017-01"
END_YM = "2025-12"

# === Utilities ===
def make_join_key(txt: str) -> str:
    if pd.isna(txt):
        return txt
    x = str(txt).lower().strip()
    x = re.sub(r"[’'`]", "", x)
    x = re.sub(r"[-_]+", " ", x)
    x = " ".join(x.split())
    x = re.sub(r"[^a-z0-9 ]+", "", x)
    return x.replace(" ", "")

def normalize_alias(d: dict) -> dict:
    return {make_join_key(k): v for k, v in d.items()}

# === Region → province mappings ===
MDG_REGION_TO_PROVINCE = normalize_alias({
    "Diana": "Antsiranana", "Sava": "Antsiranana",
    "Analanjirofo": "Toamasina", "Atsinanana": "Toamasina", "Alaotra Mangoro": "Toamasina",
    "Analamanga": "Antananarivo", "Itasy": "Antananarivo",
    "Vakinankaratra": "Antananarivo", "Bongolava": "Antananarivo",
    "Vatovavy": "Fianarantsoa", "Fitovinany": "Fianarantsoa", "Fitovivany": "Fianarantsoa",
    "Haute Matsiatra": "Fianarantsoa", "Amoron'i Mania": "Fianarantsoa",
    "Ihorombe": "Fianarantsoa", "Atsimo Atsinanana": "Fianarantsoa",
    "Atsimo-Atsinana": "Fianarantsoa", "VATOVAVY FITOVINANY": "Fianarantsoa",
    "Vatovavy Fitovinany": "Fianarantsoa",
    "Boeny": "Mahajanga", "Betsiboka": "Mahajanga", "Melaky": "Mahajanga", "Sofia": "Mahajanga",
    "Androy": "Toliara", "Anosy": "Toliara", "Atsimo Andrefana": "Toliara",
    "Atsimo-andrefana": "Toliara", "Menabe": "Toliara",
})
_MDG_PROVINCE_KEYS = {make_join_key(p) for p in
                      {"Antananarivo", "Antsiranana", "Fianarantsoa", "Mahajanga", "Toamasina", "Toliara"}}

def mdg_region_to_province_key(name: str) -> str:
    if pd.isna(name):
        return ""
    k = make_join_key(name)
    tgt = MDG_REGION_TO_PROVINCE.get(k)
    if tgt:
        return make_join_key(tgt)
    if k in _MDG_PROVINCE_KEYS:
        return k
    return ""

MOZ_REGION_TO_PROVINCE = normalize_alias({
    "Niassa": "Niassa", "Cabo Delgado": "Cabo Delgado", "Nampula": "Nampula",
    "Zambezia": "Zambezia", "Zambézia": "Zambezia",
    "Tete": "Tete", "Manica": "Manica", "Sofala": "Sofala",
    "Inhambane": "Inhambane", "Gaza": "Gaza", "Maputo": "Maputo",
    "Maputo Cidade": "Maputo", "Maputo City": "Maputo", "Maputo Provincia": "Maputo",
    "Provincia de Maputo": "Maputo", "Maputo e Matola": "Maputo",
    "Centro": "Sofala", "Sul": "Maputo", "Acolhedores": "Maputo", "Deslocados": "Tete"
})
_MOZ_PROVINCE_KEYS = {make_join_key(p) for p in
                      {"Niassa", "Cabo Delgado", "Nampula", "Zambezia",
                       "Tete", "Manica", "Sofala", "Inhambane", "Gaza", "Maputo"}}

def moz_region_to_province_key(name: str) -> str:
    if pd.isna(name):
        return ""
    k = make_join_key(name)
    tgt = MOZ_REGION_TO_PROVINCE.get(k)
    if tgt:
        return make_join_key(tgt)
    if k in _MOZ_PROVINCE_KEYS:
        return k
    return ""

# === Commodity normalization ===
def canon_commodity(s: str) -> str | None:
    s0 = str(s).strip().lower()
    s0 = re.sub(r"\s+", " ", s0)
    if s0 in ["rice", "rice (imported)", "rice (broken, imported)"]:
        return "rice"
    if s0 == "wheat flour":
        return "wheat_flour"
    if s0 in ["oil (vegetable, imported)", "vegetable oil", "oil (vegetable)", "oil (palm)"]:
        return "veg_oil"
    if s0 == "salt":
        return "salt"
    if s0 in ["maize", "maize (white)"]:
        return "maize"
    if s0 in ["cowpeas", "beans (niebe)", "beans", "niebe"]:
        return "beans_cowpea"
    return None

def normalize_unit_price(comm: str, unit: str, price: float):
    u = str(unit).strip().lower()
    if comm in ["rice", "wheat_flour", "maize", "beans_cowpea", "salt"]:
        if "kg" in u:
            m = re.search(r"([\d\.]+)\s*kg", u)
            mult = float(m.group(1)) if m else 1.0
            return "kg", price / mult
    if comm == "veg_oil" and "l" in u:
        m = re.search(r"([\d\.]+)\s*l", u)
        mult = float(m.group(1)) if m else 1.0
        return "l", price / mult
    return None, None

# === Exchange rate ===
def fetch_worldbank_fx_annual(iso2: str, indicator="PA.NUS.FCRF") -> pd.DataFrame:
    url = f"https://api.worldbank.org/v2/country/{iso2}/indicator/{indicator}?format=json&per_page=2000"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    js = r.json()
    if not isinstance(js, list) or len(js) < 2 or js[1] is None:
        return pd.DataFrame(columns=["year", "fx_lcu_per_usd"])
    rows = []
    for item in js[1]:
        val = item.get("value", None)
        yr = item.get("date", None)
        if val is not None and yr is not None:
            rows.append({"year": int(yr), "fx_lcu_per_usd": float(val)})
    return pd.DataFrame(rows)

def build_monthly_fx_from_worldbank(iso3_list, start_ym=START_YM, end_ym=END_YM,
                                    method="repeat", cache_path=FX_CACHE) -> pd.DataFrame:
    if cache_path.exists():
        return pd.read_csv(cache_path, dtype=str)

    months = pd.period_range(start_ym, end_ym, freq="M").astype(str)
    frames = []
    years = sorted({int(m[:4]) for m in months})

    for iso3 in iso3_list:
        iso2 = WB_ISO2[iso3]
        try:
            ann = fetch_worldbank_fx_annual(iso2)
        except Exception as e:
            print(f"[WARN] FX API failed for {iso3}: {e}. Using fallback.")
            ann = pd.DataFrame({"year": years, "fx_lcu_per_usd": FX_FALLBACK_LCU_PER_USD[iso3]})

        ann = ann[ann["year"].between(min(years), max(years))].copy()
        if ann.empty:
            ann = pd.DataFrame({"year": years, "fx_lcu_per_usd": FX_FALLBACK_LCU_PER_USD[iso3]})

        ann_map = dict(zip(ann["year"], ann["fx_lcu_per_usd"]))
        recs = []
        for ym in months:
            y = int(ym[:4])
            val = ann_map.get(y, FX_FALLBACK_LCU_PER_USD[iso3])
            recs.append({"country": iso3, "year_month": ym, "fx_lcu_per_usd": float(val)})
        dfm = pd.DataFrame(recs)
        frames.append(dfm)

    fxm = pd.concat(frames, ignore_index=True)
    fxm.to_csv(cache_path, index=False, encoding="utf-8-sig")
    return fxm

# === Load & harmonize one country ===
def load_and_harmonize_one(country_code: str, path: Path, fx_monthly: pd.DataFrame) -> pd.DataFrame:
    print(f"Reading {country_code}: {path}")
    df = pd.read_csv(path)

    if "Price Type" in df.columns:
        df = df[df["Price Type"].astype(str).str.lower().str.contains("retail", na=False)]

    if "Commodity" not in df.columns:
        raise ValueError(f"{country_code}: 'Commodity' not found")
    df["commodity_canon"] = df["Commodity"].apply(canon_commodity)
    df = df[~df["commodity_canon"].isna()].copy()

    date_col = "Price Date" if "Price Date" in df.columns else "Date"
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
    df = df[~df[date_col].isna()].copy()
    df["year_month"] = df[date_col].dt.strftime("%Y-%m")

    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
    df = df[~df["Price"].isna()].copy()

    unit_price = df.apply(lambda r: normalize_unit_price(r["commodity_canon"], r["Unit"], r["Price"]), axis=1)
    df["std_unit"] = [up[0] for up in unit_price]
    df["price_std"] = [up[1] for up in unit_price]
    df = df[~df["std_unit"].isna()].copy()

    a1_col = next((c for c in ["Admin 1", "Admin1", "adm1", "ADM1"] if c in df.columns), None)
    if not a1_col:
        raise ValueError(f"{country_code}: Admin1 column not found")
    df["admin1"] = df[a1_col].astype(str).str.strip()
    df["admin1_key"] = df["admin1"].apply(make_join_key)

    df["country"] = country_code
    fx_sub = fx_monthly[fx_monthly["country"] == country_code][["year_month", "fx_lcu_per_usd"]]
    fx_sub["fx_lcu_per_usd"] = fx_sub["fx_lcu_per_usd"].astype(float)
    df = df.merge(fx_sub, on="year_month", how="left")
    df["fx_lcu_per_usd"] = df["fx_lcu_per_usd"].ffill().bfill().astype(float)
    df["price_usd"] = df["price_std"] / df["fx_lcu_per_usd"]

    df["join_key"] = df["admin1_key"]
    if country_code == "MDG":
        df["join_key"] = df["admin1"].apply(mdg_region_to_province_key)
        miss = df["join_key"] == ""
        df.loc[miss, "join_key"] = df.loc[miss, "admin1_key"]
    if country_code == "MOZ":
        df["join_key"] = df["admin1"].apply(moz_region_to_province_key)
        miss = df["join_key"] == ""
        df.loc[miss, "join_key"] = df.loc[miss, "admin1_key"]

    agg = (
        df.groupby(["country", "admin1", "admin1_key", "join_key",
                    "year_month", "commodity_canon"], as_index=False)
          .agg(price_usd=("price_usd", "median"))
    )
    return agg

# === Main ===
def main():
    fx_monthly = build_monthly_fx_from_worldbank(
        iso3_list=list(IN_FILES.keys()), start_ym=START_YM, end_ym=END_YM,
        method="repeat", cache_path=FX_CACHE
    )

    frames = []
    for iso3, f in IN_FILES.items():
        if not f.exists():
            print(f"[WARN] Missing price file: {f} — skipped")
            continue
        frames.append(load_and_harmonize_one(iso3, f, fx_monthly))
    if not frames:
        raise FileNotFoundError("No usable price files found")

    prices_agg = pd.concat(frames, ignore_index=True)

    dup_mask = prices_agg.duplicated(
        subset=["country", "admin1_key", "year_month", "commodity_canon"], keep=False)
    if dup_mask.any():
        print(f"[WARN] Found {dup_mask.sum()} duplicates, averaging them")
        prices_agg = (
            prices_agg.groupby(["country", "admin1", "admin1_key", "join_key",
                                "year_month", "commodity_canon"], as_index=False)
                      .agg(price_usd=("price_usd", "mean"))
        )

    loaded_countries = sorted(prices_agg["country"].unique().tolist())
    present = (
        prices_agg.groupby(["commodity_canon", "country"]).size()
                  .unstack(fill_value=0)[loaded_countries].gt(0).all(axis=1)
    )
    common_set = set(present[present].index.tolist())
    prices_agg = prices_agg[prices_agg["commodity_canon"].isin(common_set)].copy()

    prices_wide = (
        prices_agg.pivot_table(
            index=["country", "admin1", "admin1_key", "join_key", "year_month"],
            columns="commodity_canon", values="price_usd"
        ).reset_index()
    )
    fixed = ["country", "admin1", "admin1_key", "join_key", "year_month"]
    cols = fixed + sorted([c for c in prices_wide.columns if c not in fixed])
    prices_wide = prices_wide[cols]

    prices_wide.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUT_PATH} ({len(prices_wide)} rows)")

if __name__ == "__main__":
    main()


Reading SOM: data raw\SOM-Prices-Export.csv
Reading SSD: data raw\SSD-Prices-Export.csv
Reading CAF: data raw\CAF-Prices-Export.csv
Reading MDG: data raw\MDG-Prices-Export.csv
Reading MOZ: data raw\MOZ-Prices-Export.csv
Saved: data clean\food_prices_harmonized_common_usd.csv (6094 rows)


In [8]:
!pip install pandas pandas-datareader


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# ======================================================
# build_conflict_features.py
# - Expand with IPC roster to fill all months (missing months = 0)
# ======================================================

import re
import pandas as pd
from pathlib import Path

# === Paths ===
DATA_RAW = Path("data raw")
DATA_CLEAN = Path("data clean")
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

ACLED_XLSX = DATA_RAW / "ACLED_Africa_aggregated_data.xlsx"
IPC_PATH = DATA_CLEAN / "ipc_all_clean.csv"
OUT_PATH = DATA_CLEAN / "conflict_events_admin1_monthly.csv"

# === Parameters ===
START_YM, END_YM = "2017-01", "2025-12"

COUNTRY_TO_ISO3 = {
    "Somalia": "SOM",
    "South Sudan": "SSD",
    "Central African Republic": "CAF",
    "Madagascar": "MDG",
    "Mozambique": "MOZ",
    "CAR": "CAF",
}
TARGET_ISO3 = {"SOM", "SSD", "CAF", "MDG", "MOZ"}

# === Utilities ===
def make_join_key(txt: str) -> str:
    if pd.isna(txt):
        return txt
    x = str(txt).lower().strip()
    x = re.sub(r"[’'`]", "", x)
    x = re.sub(r"[-_]+", " ", x)
    x = " ".join(x.split())
    x = re.sub(r"[^a-z0-9 ]+", "", x)
    return x.replace(" ", "")

def normalize_alias(d: dict) -> dict:
    return {make_join_key(k): v for k, v in d.items()}

def build_full_months(start_ym: str, end_ym: str):
    pr = pd.period_range(start_ym, end_ym, freq="M")
    return [f"{p.year}-{p.month:02d}" for p in pr]

# === Somalia aliases ===
SOM_ALIAS = normalize_alias({
    "Banadir": "Banadir",
    "Juba Dhexe": "Middle Juba",
    "Juba Hoose": "Lower Juba",
    "Shabelle Dhexe": "Middle Shabelle",
    "Shabelle Hoose": "Lower Shabelle",
    "jubadhexe": "Middle Juba",
    "jubahoose": "Lower Juba",
    "shabelledhexe": "Middle Shabelle",
    "shabellehoose": "Lower Shabelle",
})

def som_normalize_admin1(name: str) -> str:
    if pd.isna(name):
        return name
    k = make_join_key(name)
    return SOM_ALIAS.get(k, name)

# === MDG region → province ===
MDG_REGION_TO_PROVINCE = normalize_alias({
    "Diana": "Antsiranana", "Sava": "Antsiranana",
    "Analanjirofo": "Toamasina", "Atsinanana": "Toamasina", "Alaotra Mangoro": "Toamasina",
    "Analamanga": "Antananarivo", "Itasy": "Antananarivo", "Vakinankaratra": "Antananarivo", "Bongolava": "Antananarivo",
    "Vatovavy": "Fianarantsoa", "Fitovinany": "Fianarantsoa", "Fitovivany": "Fianarantsoa",
    "Haute Matsiatra": "Fianarantsoa", "Amoron'i Mania": "Fianarantsoa", "Ihorombe": "Fianarantsoa",
    "Atsimo Atsinanana": "Fianarantsoa", "Atsimo-Atsinana": "Fianarantsoa", "Atsimo Atsinana": "Fianarantsoa",
    "VATOVAVY FITOVINANY": "Fianarantsoa", "Vatovavy Fitovinany": "Fianarantsoa",
    "Boeny": "Mahajanga", "Betsiboka": "Mahajanga", "Melaky": "Mahajanga", "Sofia": "Mahajanga",
    "Androy": "Toliara", "Anosy": "Toliara", "Atsimo Andrefana": "Toliara", "Atsimo-andrefana": "Toliara", "Menabe": "Toliara",
})
_MDG_PROVINCE_KEYS = {make_join_key(p) for p in
                      {"Antananarivo", "Antsiranana", "Fianarantsoa", "Mahajanga", "Toamasina", "Toliara"}}

def mdg_region_to_province_key(name: str) -> str:
    if pd.isna(name):
        return ""
    k = make_join_key(name)
    prov = MDG_REGION_TO_PROVINCE.get(k)
    if prov:
        return make_join_key(prov)
    if k in _MDG_PROVINCE_KEYS:
        return k
    return ""

# === MOZ region → province ===
MOZ_REGION_TO_PROVINCE = normalize_alias({
    "Niassa": "Niassa", "Cabo Delgado": "Cabo Delgado", "Nampula": "Nampula",
    "Zambezia": "Zambezia", "Zambézia": "Zambezia",
    "Tete": "Tete", "Manica": "Manica", "Sofala": "Sofala",
    "Inhambane": "Inhambane", "Gaza": "Gaza", "Maputo": "Maputo",
    "Maputo Cidade": "Maputo", "Maputo City": "Maputo", "Maputo Provincia": "Maputo",
    "Provincia de Maputo": "Maputo", "Maputo e Matola": "Maputo",
    "Centro": "Sofala", "Sul": "Maputo", "Acolhedores": "Maputo", "Deslocados": "Tete"
})
_MOZ_PROVINCE_KEYS = {make_join_key(p) for p in
                      {"Niassa", "Cabo Delgado", "Nampula", "Zambezia",
                       "Tete", "Manica", "Sofala", "Inhambane", "Gaza", "Maputo"}}

def moz_region_to_province_key(name: str) -> str:
    if pd.isna(name):
        return ""
    k = make_join_key(name)
    prov = MOZ_REGION_TO_PROVINCE.get(k)
    if prov:
        return make_join_key(prov)
    if k in _MOZ_PROVINCE_KEYS:
        return k
    return ""

# === Column detection ===
def detect_columns(df: pd.DataFrame):
    cl = {c.lower(): c for c in df.columns}
    def pick(*names):
        for n in names:
            if n in cl:
                return cl[n]
        return None
    return {
        "country": pick("country", "country_name", "country_iso3", "iso3"),
        "admin1": pick("admin1", "admin1_name", "adm1", "region", "province", "state"),
        "events": pick("events", "event_count", "count"),
        "fatalities": pick("fatalities", "deaths"),
        "week": pick("week", "week_start", "week_date"),
        "date": pick("event_date", "date", "day"),
        "year": pick("year", "yr"),
        "month": pick("month", "mn", "mo"),
    }

def to_year_month(df: pd.DataFrame, cols: dict) -> pd.Series:
    if cols["week"] is not None:
        s = pd.to_datetime(df[cols["week"]], errors="coerce", dayfirst=True)
        return s.dt.to_period("M").astype(str)
    if cols["date"] is not None:
        s = pd.to_datetime(df[cols["date"]], errors="coerce", dayfirst=True)
        return s.dt.to_period("M").astype(str)
    if cols["year"] and cols["month"]:
        y = pd.to_numeric(df[cols["year"]], errors="coerce").astype("Int64")
        m = pd.to_numeric(df[cols["month"]], errors="coerce").astype("Int64")
        ym = (y.astype("string") + "-" + m.astype("string").str.zfill(2)).astype(object)
        ym[(y.isna()) | (m.isna())] = pd.NA
        return ym
    raise ValueError("Cannot find valid date columns")

# === Main ===
def main():
    print("————————————————————————————————————————")
    print("Building conflict features (ACLED → IPC admin1)")
    print("————————————————————————————————————————")

    if not ACLED_XLSX.exists():
        raise FileNotFoundError(f"File not found: {ACLED_XLSX}")

    sheets = pd.read_excel(ACLED_XLSX, sheet_name=None, engine="openpyxl")
    all_rows = []

    for sheet_name, df in sheets.items():
        if df is None or df.empty:
            continue
        cols = detect_columns(df)
        if cols["country"] is None or cols["admin1"] is None:
            continue

        work = df.copy()
        try:
            work["year_month"] = to_year_month(work, cols)
        except Exception:
            continue
        work = work.dropna(subset=["year_month"])

        months = set(build_full_months(START_YM, END_YM))
        work = work[work["year_month"].isin(months)].copy()
        if work.empty:
            continue

        work["country_raw"] = work[cols["country"]].astype(str).str.strip()
        work["country_iso3"] = work["country_raw"].map(COUNTRY_TO_ISO3).fillna(work["country_raw"])
        work = work[work["country_iso3"].isin(TARGET_ISO3)].copy()
        if work.empty:
            continue

        work["admin1"] = work[cols["admin1"]].astype(str).str.strip()
        is_som = work["country_iso3"] == "SOM"
        work.loc[is_som, "admin1"] = work.loc[is_som, "admin1"].apply(som_normalize_admin1)

        work["events"] = pd.to_numeric(work[cols["events"]], errors="coerce").fillna(0)
        work["fatalities"] = pd.to_numeric(work[cols["fatalities"]], errors="coerce").fillna(0)

        work["admin1_key"] = work["admin1"].apply(make_join_key)
        work["join_key"] = work["admin1_key"]

        mdg_mask = work["country_iso3"] == "MDG"
        moz_mask = work["country_iso3"] == "MOZ"
        work.loc[mdg_mask, "join_key"] = work.loc[mdg_mask, "admin1"].map(mdg_region_to_province_key)
        work.loc[moz_mask, "join_key"] = work.loc[moz_mask, "admin1"].map(moz_region_to_province_key)

        fail_mask = (work["join_key"] == "")
        work.loc[fail_mask, "join_key"] = work.loc[fail_mask, "admin1_key"]

        agg = (
            work.groupby(["country_iso3", "join_key", "year_month"], as_index=False)
            .agg(conflict_events=("events", "sum"), fatalities=("fatalities", "sum"))
        )
        all_rows.append(agg)

    if not all_rows:
        raise RuntimeError("No valid ACLED data found")

    conflict = pd.concat(all_rows, ignore_index=True).rename(columns={"country_iso3": "country"})

    ipc = pd.read_csv(IPC_PATH, dtype=str)
    ipc["admin1_key"] = ipc["admin1"].apply(make_join_key)
    ipc["join_key"] = ipc["admin1_key"]

    mdg_mask = ipc["country"] == "MDG"
    moz_mask = ipc["country"] == "MOZ"
    ipc.loc[mdg_mask, "join_key"] = ipc.loc[mdg_mask, "admin1"].map(mdg_region_to_province_key)
    ipc.loc[moz_mask, "join_key"] = ipc.loc[moz_mask, "admin1"].map(moz_region_to_province_key)

    roster = ipc[ipc["country"].isin(list(TARGET_ISO3))][["country", "admin1", "admin1_key", "join_key"]].drop_duplicates()

    all_months = build_full_months(START_YM, END_YM)
    frame = pd.DataFrame(
        [{"country": r["country"], "admin1": r["admin1"], "admin1_key": r["admin1_key"],
          "join_key": r["join_key"], "year_month": ym}
         for _, r in roster.iterrows() for ym in all_months]
    )

    out = frame.merge(conflict, on=["country", "join_key", "year_month"], how="left")
    out["conflict_events"] = out["conflict_events"].fillna(0).astype(int)
    out["fatalities"] = out["fatalities"].fillna(0).astype(int)

    out = (
        out.groupby(["country", "admin1", "admin1_key", "year_month"], as_index=False)
        .agg(conflict_events=("conflict_events", "sum"), fatalities=("fatalities", "sum"))
    )

    out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUT_PATH} ({len(out)} rows)")

    for c in sorted(TARGET_ISO3):
        sub = out[out["country"] == c]
        if len(sub):
            print(f"{c}: admin1={sub['admin1'].nunique()}, months={sub['year_month'].nunique()}, avg_events={sub['conflict_events'].mean():.2f}")

if __name__ == "__main__":
    main()


————————————————————————————————————————
Building conflict features (ACLED → IPC admin1)
————————————————————————————————————————
Saved: data clean\conflict_events_admin1_monthly.csv (11988 rows)
CAF: admin1=29, months=108, avg_events=1.19
MDG: admin1=21, months=108, avg_events=4.19
MOZ: admin1=20, months=108, avg_events=4.15
SOM: admin1=27, months=108, avg_events=10.60
SSD: admin1=14, months=108, avg_events=8.29


In [10]:
# ======================================================
# build_training_dataset.py — unified version (5 countries)
# Combine IPC (label) + Rainfall + Conflict + Food Prices (USD)
# ======================================================

import re
import pandas as pd
from pathlib import Path

# ---------- Paths ----------
DATA_CLEAN = Path("data clean")
IPC_PATH = DATA_CLEAN / "ipc_all_clean.csv"
RAIN_PATH = DATA_CLEAN / "chirps_all_admin1_monthly_harmonized.csv"
CONFLICT_PATH = DATA_CLEAN / "conflict_events_admin1_monthly.csv"
FOOD_PATH = DATA_CLEAN / "food_prices_harmonized_common_usd.csv"

OUT_PATH = DATA_CLEAN / "train_features_2017_2025.csv"
COVER_PATH = DATA_CLEAN / "feature_coverage_report.csv"
MISSING_PATH = DATA_CLEAN / "feature_missing_admin1.csv"

# ---------- Parameters ----------
TARGET_COUNTRIES = {"SOM", "SSD", "CAF", "MDG", "MOZ"}
START_YM, END_YM = "2017-01", "2025-12"

# ---------- Utilities ----------
def make_join_key(txt: str) -> str:
    if pd.isna(txt):
        return txt
    x = str(txt).lower().strip()
    x = re.sub(r"[’'`]", "", x)
    x = re.sub(r"[-_]+", " ", x)
    x = " ".join(x.split())
    x = re.sub(r"[^a-z0-9 ]+", "", x)
    return x.replace(" ", "")

def normalize_alias(d: dict) -> dict:
    return {make_join_key(k): v for k, v in d.items()}

# ---------- Somalia aliases ----------
SOM_ALIAS_RAW = {
    "Banadir": "Banadir",
    "Juba Dhexe": "Middle Juba",
    "Juba Hoose": "Lower Juba",
    "Shabelle Dhexe": "Middle Shabelle",
    "Shabelle Hoose": "Lower Shabelle",
    "Woqooyi Galbeed": "Woqooyi Galbeed",
    "Waqooyi Galbeed": "Woqooyi Galbeed",
}
SOM_ALIAS = {make_join_key(k): make_join_key(v) for k, v in SOM_ALIAS_RAW.items()}

def apply_som_alias_key(k: str) -> str:
    return SOM_ALIAS.get(k, k)

# ---------- MDG region → province ----------
MDG_REGION_TO_PROVINCE = normalize_alias({
    "Diana": "Antsiranana", "Sava": "Antsiranana",
    "Analanjirofo": "Toamasina", "Atsinanana": "Toamasina", "Alaotra Mangoro": "Toamasina",
    "Analamanga": "Antananarivo", "Itasy": "Antananarivo", "Vakinankaratra": "Antananarivo", "Bongolava": "Antananarivo",
    "Vatovavy": "Fianarantsoa", "Fitovinany": "Fianarantsoa", "Fitovivany": "Fianarantsoa",
    "Haute Matsiatra": "Fianarantsoa", "Amoron'i Mania": "Fianarantsoa", "Ihorombe": "Fianarantsoa",
    "Atsimo Atsinanana": "Fianarantsoa", "Atsimo-Atsinana": "Fianarantsoa", "Atsimo Atsinana": "Fianarantsoa",
    "Vatovavy Fitovinany": "Fianarantsoa",
    "Boeny": "Mahajanga", "Betsiboka": "Mahajanga", "Melaky": "Mahajanga", "Sofia": "Mahajanga",
    "Androy": "Toliara", "Anosy": "Toliara", "Atsimo Andrefana": "Toliara", "Menabe": "Toliara",
})
_MDG_PROVINCE_KEYS = {make_join_key(p) for p in
                      {"Antananarivo","Antsiranana","Fianarantsoa","Mahajanga","Toamasina","Toliara"}}

def mdg_region_to_province_key(name: str) -> str:
    if pd.isna(name):
        return ""
    k = make_join_key(name)
    prov = MDG_REGION_TO_PROVINCE.get(k)
    if prov:
        return make_join_key(prov)
    if k in _MDG_PROVINCE_KEYS:
        return k
    return ""

# ---------- MOZ region → province ----------
MOZ_REGION_TO_PROVINCE = normalize_alias({
    "Niassa": "Niassa", "Cabo Delgado": "Cabo Delgado", "Nampula": "Nampula",
    "Zambezia": "Zambezia", "Zambézia": "Zambezia",
    "Tete": "Tete", "Manica": "Manica", "Sofala": "Sofala",
    "Inhambane": "Inhambane", "Gaza": "Gaza", "Maputo": "Maputo",
    "Maputo City": "Maputo", "Maputo Provincia": "Maputo",
})
_MOZ_PROVINCE_KEYS = {make_join_key(p) for p in
                      {"Niassa", "Cabo Delgado", "Nampula", "Zambezia", "Tete",
                       "Manica", "Sofala", "Inhambane", "Gaza", "Maputo"}}

def moz_region_to_province_key(name: str) -> str:
    if pd.isna(name):
        return ""
    k = make_join_key(name)
    prov = MOZ_REGION_TO_PROVINCE.get(k)
    if prov:
        return make_join_key(prov)
    if k in _MOZ_PROVINCE_KEYS:
        return k
    return ""

def apply_country_join_key(df: pd.DataFrame, country_col="country", admin_col="admin1"):
    """Return a df with join_key aligned for SOM, MDG, MOZ."""
    df["admin1_key"] = df[admin_col].apply(make_join_key)
    df["join_key"] = df["admin1_key"]
    som_mask = df[country_col] == "SOM"
    mdg_mask = df[country_col] == "MDG"
    moz_mask = df[country_col] == "MOZ"
    df.loc[som_mask, "join_key"] = df.loc[som_mask, "admin1_key"].apply(apply_som_alias_key)
    df.loc[mdg_mask, "join_key"] = df.loc[mdg_mask, admin_col].map(mdg_region_to_province_key)
    df.loc[moz_mask, "join_key"] = df.loc[moz_mask, admin_col].map(moz_region_to_province_key)
    df["join_key"] = df["join_key"].fillna(df["admin1_key"])
    return df

def safe_deduplicate(df, key_cols, agg_dict):
    dup_mask = df.duplicated(subset=key_cols, keep=False)
    if dup_mask.any():
        print(f"[WARN] Found {dup_mask.sum()} duplicates on {key_cols}, aggregating...")
        return df.groupby(key_cols, as_index=False).agg(agg_dict)
    return df

# ---------- Main ----------
def main():
    # === IPC ===
    ipc = pd.read_csv(IPC_PATH, dtype=str)
    ipc = ipc[ipc["country"].isin(TARGET_COUNTRIES)]
    ipc = ipc[(ipc["year_month"] >= START_YM) & (ipc["year_month"] <= END_YM)]
    ipc = apply_country_join_key(ipc)
    label_cols = ["phase3plus_binary_20pct", "affected_pct_3p", "affected_num_3p"]
    cols_keep = ["country", "admin1", "admin1_key", "join_key", "year_month", "admin1_pop_est"] + [
        c for c in label_cols if c in ipc.columns
    ]
    base = ipc[cols_keep].drop_duplicates()
    base = safe_deduplicate(base, ["country", "join_key", "year_month"],
                            {c: "first" for c in base.columns if c not in ["country", "join_key", "year_month"]})

    # === Rainfall ===
    rain = pd.read_csv(RAIN_PATH, dtype=str)
    rain = rain[rain["country"].isin(TARGET_COUNTRIES)]
    rain = apply_country_join_key(rain)
    rain["rain_mm"] = pd.to_numeric(rain["rain_mm"], errors="coerce")
    rain = safe_deduplicate(rain, ["country", "join_key", "year_month"], {"rain_mm": "mean"})

    # === Conflict ===
    conf = pd.read_csv(CONFLICT_PATH, dtype=str)
    conf = conf[conf["country"].isin(TARGET_COUNTRIES)]
    conf = apply_country_join_key(conf)
    for col in ["conflict_events", "fatalities"]:
        if col in conf.columns:
            conf[col] = pd.to_numeric(conf[col], errors="coerce").fillna(0).astype(int)
    conf = safe_deduplicate(conf, ["country", "join_key", "year_month"],
                            {"conflict_events": "sum", "fatalities": "sum"})

    # === Food Prices ===
    food = pd.read_csv(FOOD_PATH, dtype=str)
    food = food[food["country"].isin(TARGET_COUNTRIES)]
    food = apply_country_join_key(food)
    feature_cols_food = [c for c in food.columns if c not in {"country", "admin1", "admin1_key", "join_key", "year_month"}]
    for c in feature_cols_food:
        food[c] = pd.to_numeric(food[c], errors="coerce")
    food = safe_deduplicate(food, ["country", "join_key", "year_month"], {c: "mean" for c in feature_cols_food})

    # === Merge all ===
    df = base.merge(rain, on=["country", "join_key", "year_month"], how="left")
    df = df.merge(conf, on=["country", "join_key", "year_month"], how="left")
    df = df.merge(food, on=["country", "join_key", "year_month"], how="left")

    # === Coverage summary ===
    rain_cols = ["rain_mm"]
    conflict_cols = ["conflict_events", "fatalities"]
    food_cols = [c for c in df.columns if c.startswith("rice") or c.startswith("maize") or c.startswith("veg_oil") or c.startswith("wheat")]
    cov_rows = []
    for c in sorted(TARGET_COUNTRIES):
        sub = df[df["country"] == c]
        for name, cols in [("rain", rain_cols), ("conflict", conflict_cols), ("food", food_cols)]:
            cov = float((sub[cols].notna().mean()).mean()) if cols else 1.0
            cov_rows.append({"country": c, "feature_group": name, "coverage": round(cov, 4)})
    cov_pivot = pd.DataFrame(cov_rows).pivot(index="feature_group", columns="country", values="coverage").reset_index()

    # === Missing admin1 list ===
    miss_mask = df[rain_cols + conflict_cols + food_cols].isna().any(axis=1)
    missing_admin1 = df.loc[miss_mask, ["country", "admin1", "join_key"]].drop_duplicates()

    # === Save ===
    df = df.sort_values(["country", "admin1", "year_month"]).reset_index(drop=True)
    df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
    cov_pivot.to_csv(COVER_PATH, index=False, encoding="utf-8-sig")
    missing_admin1.to_csv(MISSING_PATH, index=False, encoding="utf-8-sig")

    print(f"Saved merged dataset: {OUT_PATH} ({len(df)} rows)")
    print("Coverage summary:")
    print(cov_pivot.to_string(index=False))

if __name__ == "__main__":
    main()


[WARN] Found 5076 duplicates on ['country', 'join_key', 'year_month'], aggregating...
[WARN] Found 5076 duplicates on ['country', 'join_key', 'year_month'], aggregating...
[WARN] Found 5076 duplicates on ['country', 'join_key', 'year_month'], aggregating...
[WARN] Found 863 duplicates on ['country', 'join_key', 'year_month'], aggregating...
Saved merged dataset: data clean\train_features_2017_2025.csv (8208 rows)
Coverage summary:
feature_group   CAF    MDG    MOZ    SOM    SSD
     conflict 1.000 1.0000 1.0000 1.0000 1.0000
         food 0.461 0.2963 0.6507 0.7716 0.5756
         rain 1.000 1.0000 1.0000 1.0000 1.0000


In [11]:
# ======================================================
# build_lagged_horizon_features.py
# ======================================================

import re
import pandas as pd
from pathlib import Path

# ---------- Paths ----------
DATA_CLEAN  = Path("data clean")
INPUT_PATH  = DATA_CLEAN / "train_features_2017_2025.csv"
OUTPUT_PATH = DATA_CLEAN / "train_features_2017_2025_lagged.csv"

# ---------- 1) Load & Validate ----------
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"File not found: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH)
print(f"Loaded: {INPUT_PATH}  → {df.shape[0]:,} rows × {df.shape[1]} cols")

required_cols = ["country", "admin1", "admin1_key", "year_month"]
missing = set(required_cols) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ---------- 2) Type conversion ----------
df["year_month"] = pd.to_datetime(df["year_month"], format="%Y-%m", errors="coerce")

label_col = "phase3plus_binary_20pct"
id_cols = ["country", "admin1", "admin1_key", "year_month"]

# Exclude non-feature columns (IDs + join_key + label)
exclude_cols = set(id_cols + [label_col, "join_key"])
candidate_cols = [c for c in df.columns if c not in exclude_cols]

# Convert to numeric if possible
for c in candidate_cols + [label_col]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# List of numeric features (post-coercion)
feature_cols = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(df[c])]
print(f"Feature columns ({len(feature_cols)}): {', '.join(feature_cols[:8])}...")

# ---------- 3) Canonical admin1 name ----------
def most_frequent_name(s: pd.Series):
    s_clean = s.dropna().astype(str).str.strip()
    return s_clean.value_counts().idxmax() if not s_clean.empty else None

canon_name = (
    df.groupby(["country", "admin1_key"])["admin1"]
      .agg(most_frequent_name)
      .rename("admin1_canon")
      .reset_index()
)

df = df.drop(columns=["admin1"], errors="ignore").merge(
    canon_name, on=["country", "admin1_key"], how="left"
).rename(columns={"admin1_canon": "admin1"})

# ---------- 4) Deduplicate safety check ----------
dup_count = df.duplicated(subset=["country", "admin1_key", "year_month"]).sum()
if dup_count > 0:
    print(f"[WARN] Found {dup_count} duplicates. Averaging numeric columns...")
    num_cols = [c for c in df.columns if c not in id_cols + ["admin1"]]
    df = (
        df.groupby(["country", "admin1", "admin1_key", "year_month"], as_index=False)[num_cols]
          .mean()
    )
else:
    print("No duplicates found.")

# ---------- 5) Impute food price bases BEFORE building lags ----------
FOOD_BASES = [c for c in ["beans_cowpea", "maize", "rice", "veg_oil", "wheat_flour"] if c in df.columns]
if FOOD_BASES:
    print(f"Imputing food price bases: {FOOD_BASES}")

    # 5.1 forward/backward fill within (country, admin1_key)
    def ts_fill(g):
        g = g.sort_values("year_month").copy()
        for col in FOOD_BASES:
            g[col] = g[col].ffill().bfill()
        return g

    df = df.groupby(["country", "admin1_key"], group_keys=False).apply(ts_fill)

    # 5.2 country × month median (preserve seasonality)
    med = (
        df.groupby(["country", "year_month"])[FOOD_BASES]
          .median().reset_index()
          .rename(columns={c: f"{c}__cy_median" for c in FOOD_BASES})
    )
    df = df.merge(med, on=["country", "year_month"], how="left")
    for col in FOOD_BASES:
        df[col] = df[col].fillna(df[f"{col}__cy_median"])
        df.drop(columns=[f"{col}__cy_median"], inplace=True)

    # 5.3 global mean fallback (final)
    for col in FOOD_BASES:
        df[col] = df[col].fillna(df[col].mean())

else:
    print("No food price columns found; skipping imputation.")

# ---------- 6) Build lags & horizon labels ----------
def add_lags(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("year_month").copy()
    for col in feature_cols:
        if col in group.columns:
            group[f"{col}_lag1"] = group[col].shift(1)
            group[f"{col}_lag2"] = group[col].shift(2)
            group[f"{col}_lag3"] = group[col].shift(3)
    if label_col in group.columns:
        group[f"{label_col}_h1"] = group[label_col].shift(-1)
        group[f"{label_col}_h3"] = group[label_col].shift(-3)
    return group

df_lagged = df.groupby(["country", "admin1_key"], group_keys=False).apply(add_lags)

# ---------- 7) Trim rows without H3 label ----------
label_h3 = f"{label_col}_h3"
if label_h3 in df_lagged.columns:
    before = len(df_lagged)
    df_lagged = df_lagged.dropna(subset=[label_h3])
    after = len(df_lagged)
    print(f"Dropped {before - after} rows without {label_h3}")

# ---------- 8) Final ordering & save ----------
df_lagged["year_month"] = df_lagged["year_month"].dt.strftime("%Y-%m")
df_lagged = df_lagged.sort_values(["country", "admin1", "year_month"]).reset_index(drop=True)

id_cols_final = ["country", "admin1", "admin1_key", "year_month"]
label_cols = [label_col]
feature_cols_final = [c for c in df_lagged.columns if c in feature_cols]
lag_cols = [c for c in df_lagged.columns if "_lag" in c]
horizon_cols = [f"{label_col}_h1", f"{label_col}_h3"]

ordered_cols = id_cols_final + label_cols + feature_cols_final + lag_cols + horizon_cols
df_lagged = df_lagged[[c for c in ordered_cols if c in df_lagged.columns]]

df_lagged.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Exported: {OUTPUT_PATH}")
print(f"Shape: {df_lagged.shape[0]:,} rows × {df_lagged.shape[1]} cols")
print("Done: canonical names, deduped, imputed food prices, lagged features, and h1/h3 labels.")


Loaded: data clean\train_features_2017_2025.csv  → 8,208 rows × 16 cols
Feature columns (10): admin1_pop_est, affected_pct_3p, affected_num_3p, rain_mm, conflict_events, fatalities, beans_cowpea, maize...
No duplicates found.
Imputing food price bases: ['beans_cowpea', 'maize', 'rice', 'veg_oil']


C:\Users\migo8\AppData\Local\Temp\ipykernel_29072\2185454954.py:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["country", "admin1_key"], group_keys=False).apply(ts_fill)
C:\Users\migo8\AppData\Local\Temp\ipykernel_29072\2185454954.py:118: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_lagged = df.groupby(["country", "admin1_key"], group_keys=False).apply(add_lags)


Dropped 228 rows without phase3plus_binary_20pct_h3
Exported: data clean\train_features_2017_2025_lagged.csv
Shape: 7,980 rows × 47 cols
Done: canonical names, deduped, imputed food prices, lagged features, and h1/h3 labels.
